# BraTS / UCSF Error Analysis

## 1. Imports & Configuration

In [ ]:
import os
import re
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from scipy import stats
from scipy.stats import gaussian_kde, wasserstein_distance
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List
from itertools import combinations

# Modalities and label definitions
MODALITIES   = ["t1n", "t2w", "t2f", "t1c"]
LABEL_MAP    = {1: "NCR", 2: "Edema", 3: "ET"}
TUMOR_LABELS = list(LABEL_MAP.keys())

# Failure analysis threshold and colors
DICE_FAIL_THRESH = 0.3
FAIL_COLOR       = "#e74c3c"   # red
PASS_COLOR       = "#29b93f"   # green

# Output directory
OUTPUT_DIR = "dataset_error_diagnostics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Split colors for cross-dataset comparison plots
SPLITS = [
    ("Training",   "#4C72B0"),
    ("Test",       "#DD8452"),
    ("UCSF",       "#C44E52"),
]

## 2. Dataset Configuration
##### defined class primarily because UCSF and BraTS have different naming and labling conventions

In [ ]:
@dataclass
class DatasetConfig:
    root: str
    modality_map: Dict[str, str] = field(default_factory=lambda: {
        "t1n": "t1n", 
        "t2w": "t2w", 
        "t2f": "t2f", 
        "t1c": "t1c"
    })
    seg_folder: str = "seg"
    label_remap: Dict[int, int] = field(default_factory=dict)

# paths to BraTS data
TRAIN_ROOT      = "/path/to/your/brats/training/data"
VALIDATION_ROOT = "/path/to/your/brats/validation/data"
TEST_ROOT       = "/path/to/your/brats/test/data"

# set BraTS config
BRATS_TRAIN_CFG = DatasetConfig(root=TRAIN_ROOT)
BRATS_VAL_CFG   = DatasetConfig(root=VALIDATION_ROOT)
BRATS_TEST_CFG  = DatasetConfig(root=TEST_ROOT)

# create the UCSF config (didnt set root because not really needed later)
UCSF_CFG = DatasetConfig(
    root="/path/to/your/UCSF/data",
    modality_map={
            "T1": "t1n", 
            "T2": "t2w", 
            "FLAIR": "t2f", 
            "T1c": "t1c"},
    seg_folder="tumor_segmentation",
    label_remap={4: 3},   # remap UCSF label 4 (ET) --> BraTS label 3
)

# paths to segmentation result JSON files
VAL_JSON  = "/path/to/your/validation/segmentation/scores/*.json"
TEST_JSON = "/path/to/your/test/segmentation/scores/*.json"
UCSF_JSON = "/path/to/your/UCSF/segmentation/scores/*.json"


## 3. Data Loading

In [ ]:
# load png slices (convert to L from RBG just in case)
def load_slices_from_folder(folder: str) -> np.ndarray | None:
    if not os.path.isdir(folder):
        return None
    paths = sorted(glob.glob(os.path.join(folder, "*.png")))
    if not paths:
        return None
    return np.stack(
        [np.array(Image.open(p).convert("L"), dtype=np.float32) for p in paths]
    )

# load segmentation masks 
def load_mask_slices_from_folder(folder: str) -> np.ndarray | None:
    if not os.path.isdir(folder):
        return None
    paths = sorted(glob.glob(os.path.join(folder, "*.png")))
    if not paths:
        return None
    return np.stack(
        [np.array(Image.open(p), dtype=np.int32) for p in paths]
    )

# remap seg labels if needed (i.e., UCSF to BraTS convention) 
def _remap_labels(seg: np.ndarray, remap: Dict[int, int]) -> np.ndarray:
    out = seg.copy()
    for src, dst in remap.items():
        out[seg == src] = dst
    return out

#  function for loading dataset splits (train/test/UCSF)
def load_split(cfg: DatasetConfig, patient_ids=None) -> dict:
    all_dirs = sorted(
        d for d in glob.glob(os.path.join(cfg.root, "*")) if os.path.isdir(d)
    )
    if patient_ids is not None:
        id_set   = set(patient_ids)
        all_dirs = [d for d in all_dirs if os.path.basename(d) in id_set]

    cases = {}
    for pdir in all_dirs:
        pid  = os.path.basename(pdir)
        case = {"name": pid}

        for raw_folder, canonical_key in cfg.modality_map.items():
            arr = load_slices_from_folder(os.path.join(pdir, raw_folder))
            case.setdefault(canonical_key, arr)
        for key in MODALITIES:
            case.setdefault(key, None)

        raw_seg     = load_mask_slices_from_folder(os.path.join(pdir, cfg.seg_folder))
        case["seg"] = _remap_labels(raw_seg, cfg.label_remap) if raw_seg is not None else None
        cases[pid]  = case

    print(f"  Loaded {len(cases)} patients from {cfg.root!r}")
    return cases

# merge train and val for the Dice Score analysis
def load_multi_root(cfgs: List[DatasetConfig], patient_ids=None) -> dict:
    merged = {}
    for cfg in cfgs:
        merged.update(load_split(cfg, patient_ids=patient_ids))
    print(f"  -> {len(merged)} patients total across {len(cfgs)} roots")
    return merged


##### load all the datasets

In [ ]:
train_cases      = load_split(BRATS_TRAIN_CFG)
validation_cases = load_split(BRATS_VAL_CFG)
test_cases       = load_split(BRATS_TEST_CFG)

# since cross-validated results used the combined original BraTS train and validation sets 
trainval_cases   = {**train_cases, **validation_cases}

ucsf_cases = load_split(UCSF_CFG)

print("\nAll datasets loaded ✓")


## 4. Image Quality Metrics

In [ ]:

# Signal to noise ratio (SNR)
def estimate_snr(volume: np.ndarray) -> float:
    snrs = []
    for slc in volume:
        signal = slc[slc > 0]            # to ignore background pixels
        if len(signal) < 10:
            continue
        noise = signal.std()
        if noise > 0:
            snrs.append(signal.mean() / noise)
    return float(np.mean(snrs)) if snrs else np.nan

# Contrast to noise ratio (CNR) --> only considering slices with tumor tissue
def estimate_cnr(volume: np.ndarray, seg: np.ndarray) -> float:
 #
    # slices with no tumor are skipped
    cnrs = []
    for slc, seg_slc in zip(volume, seg):          # segmentation masks necessary 
        tumor_mask  = np.isin(seg_slc, TUMOR_LABELS)
        tissue_mask = (slc > 0) & ~tumor_mask
        tumor_px    = slc[tumor_mask]
        tissue_px   = slc[tissue_mask]
        if len(tumor_px) < 10 or len(tissue_px) < 10:
            continue
        pooled_std = np.sqrt((tumor_px.std()**2 + tissue_px.std()**2) / 2)
        if pooled_std > 0:
            cnrs.append(abs(tumor_px.mean() - tissue_px.mean()) / pooled_std)
    return float(np.mean(cnrs)) if cnrs else np.nan


# Creating a registry for the metrics to simplify plotting
METRIC_REGISTRY = [
    ("SNR", estimate_snr, False, "SNR",  "image_quality_snr.png"),
    ("CNR", estimate_cnr, True,  "CNR (tumor vs tissue)", "image_quality_cnr.png"),
]

def compute_metrics(image_cases: dict) -> dict:

    results = {}
    for pid, case in image_cases.items():
        row = {}
        seg = case.get("seg")
        for mod in MODALITIES:
            vol = case.get(mod)
            if vol is None:
                continue
            for metric_name, fn, needs_seg, *_ in METRIC_REGISTRY:
                key      = f"{metric_name}_{mod}"
                row[key] = fn(vol, seg) if needs_seg and seg is not None                            else (fn(vol) if not needs_seg else np.nan)
        results[pid] = row
    return results


## 5. Cross-Dataset Image Quality Comparison

In [ ]:

def _wasserstein_between_datasets(vals_per_split, split_names, metric, modality):
    print(f"[{metric} — {modality.upper()}] Pairwise Wasserstein W1:")
    for (s1, d1), (s2, d2) in combinations(zip(split_names, vals_per_split), 2):
        if len(d1) < 2 or len(d2) < 2:
            print(f" {s1} vs {s2}: insufficient data")
            continue
        d1_arr, d2_arr = np.array(d1), np.array(d2)
        w1 = wasserstein_distance(d1_arr, d2_arr)
        pooled_std = np.sqrt((d1_arr.std()**2 + d2_arr.std()**2) / 2)
        w1_norm = w1 / pooled_std if pooled_std > 0 else np.nan
        print(f"{s1} vs {s2}: W1={w1:.3f}  W1_norm={w1_norm:.3f}")

# helper function for creating boxplots
def _boxplot(ax, all_vals, split_names, colors, title, ylabel):
    bp = ax.boxplot(all_vals, labels=split_names, patch_artist=True,
                    medianprops=dict(color="red"))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    for i, (data, color) in enumerate(zip(all_vals, colors), start=1):
        jitter = np.random.normal(0, 0.05, size=len(data))         # improve visualization
        ax.scatter(np.full(len(data), i) + jitter, data, alpha=0.4, s=15, color=color)
    ax.set_title(title, fontsize=9)
    ax.set_ylabel(ylabel)

# helper function for plotting image quality metrics
def plot_image_quality_metrics(*all_cases, modalities=MODALITIES):
    pairs = list(zip(SPLITS[:len(all_cases)], all_cases))
    split_names = [s for (s, _),  _ in pairs]
    colors = [col for (_, col), _ in pairs]

    def compute_values(fn, needs_seg, cases, mod):
        if needs_seg:
            return [fn(c[mod], c["seg"]) for c in cases.values()
                    if c.get(mod) is not None and c.get("seg") is not None]
        return [fn(c[mod]) for c in cases.values() if c.get(mod) is not None]

    # to clean oujt NaN values
    def clean(values):
        return [v for v in values if not np.isnan(v)]

    metric_data = {
        key: {
            mod: [clean(compute_values(fn, needs_seg, cases, mod))
                  for _, cases in pairs]
            for mod in modalities
        }
        for key, fn, needs_seg, *_ in METRIC_REGISTRY
    }

    # plot one figure per metrics
    for key, _, _, ylabel, fname in METRIC_REGISTRY:
        fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey=False)
        fig.suptitle(f"{key} — All Modalities & Splits", fontsize=13, fontweight="bold")

        for ax, mod in zip(axes.flatten(), modalities):
            vals = metric_data[key][mod]
            _boxplot(ax, vals, split_names, colors, title=mod.upper(), ylabel=ylabel)

            print(f"{key} — {mod.upper()}:")
            for name, data in zip(split_names, vals):
                if data:
                    arr = np.array(data)
                    print(f"{name:12s}: mean={arr.mean():.3f}  "
                          f"std={arr.std():.3f}  skew={stats.skew(arr):+.3f}")
            _wasserstein_between_datasets(vals, split_names, key, mod)

        for ax in axes.flatten()[len(modalities):]:
            ax.set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150)
        plt.show()
        plt.close()
        print(f"Saved {os.path.join(OUTPUT_DIR, fname)}")


In [ ]:
plot_image_quality_metrics(train_cases, validation_cases, test_cases, ucsf_cases)

## 6. Image Quality Analysis — Catastrophic Failure Cases (Dice < 0.3)
Comparing failed vs. passing cases on image quality metrics.

In [ ]:
# begin by defining funcs for sorting through results files and aggregating cases by patient ID

def patient_id_from_path(filepath: str) -> str:
    name = os.path.splitext(os.path.basename(filepath))[0]
    return re.sub(r"_\d+$", "", name)


def aggregate_cases(json_path: str) -> dict:

    with open(json_path) as f:
        raw = json.load(f)

    accum = defaultdict(lambda: defaultdict(
        lambda: {"TP": 0.0, "FP": 0.0, "FN": 0.0, "n_ref": 0.0, "n_pred": 0.0}
    ))
    for case in raw["metric_per_case"]:
        pid = patient_id_from_path(case["reference_file"])
        for label_str, m in case["metrics"].items():
            lb = int(label_str)
            for k in ["TP", "FP", "FN", "n_ref", "n_pred"]:
                accum[pid][lb][k] += m[k]

    results = {}
    for pid, labels in accum.items():
        row       = {}
        dice_vals = []
        for label, name in LABEL_MAP.items():
            tp, fp, fn = labels[label]["TP"], labels[label]["FP"], labels[label]["FN"]
            denom      = 2 * tp + fp + fn
            dice       = (2 * tp / denom) if denom > 0 else np.nan
            row[f"Dice_{name}"]  = dice
            row[f"n_ref_{name}"] = labels[label]["n_ref"]
            if not np.isnan(dice):
                dice_vals.append(dice)
        row["Dice_mean"] = float(np.mean(dice_vals)) if dice_vals else np.nan
        results[pid] = row

    print(f"  {len(results)} patients from {os.path.basename(json_path)}")
    return results

# Next, divide between passing and failed cases (i.e., Dice < 0.3) 
def split_by_dice(dice_cases: dict, label_name="Dice_mean", threshold=DICE_FAIL_THRESH):

    failed  = [pid for pid, v in dice_cases.items()
               if not np.isnan(v.get(label_name, np.nan)) and v[label_name] < threshold]
    passing = [pid for pid, v in dice_cases.items()
               if not np.isnan(v.get(label_name, np.nan)) and v[label_name] >= threshold]

    print(f"  {label_name}: {len(failed)} failed (<{threshold}), "
          f"{len(passing)} passing (>={threshold})")
    return failed, passing


##### creating plotting helper functions specific to pass/fail analysis

In [ ]:
def _boxplot_fail_pass(ax, fail_vals, pass_vals, title, ylabel):
    data   = [fail_vals, pass_vals]
    labels = [f"Dice<{DICE_FAIL_THRESH}\n(n={len(fail_vals)})",
              f"Dice>={DICE_FAIL_THRESH}\n(n={len(pass_vals)})"]
    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))
    for patch, c in zip(bp["boxes"], [FAIL_COLOR, PASS_COLOR]):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)
    for i, (d, c) in enumerate(zip(data, [FAIL_COLOR, PASS_COLOR]), start=1):
        jitter = np.random.normal(0, 0.06, len(d))
        ax.scatter(np.full(len(d), i) + jitter, d, alpha=0.4, s=12, color=c)
    if len(fail_vals) >= 2 and len(pass_vals) >= 2:
        _, p = stats.mannwhitneyu(fail_vals, pass_vals, alternative="two-sided")
        ax.set_title(f"{title}\nMann-Whitney p={p:.4f}", fontsize=9)
    else:
        ax.set_title(title, fontsize=9)
    ax.set_ylabel(ylabel)

def plot_quality_fail_vs_pass(fail_metrics, pass_metrics, split_name, label_name):
    for metric_name, _, _, *_ in METRIC_REGISTRY:
        n_mods = len(MODALITIES)
        fig, axes = plt.subplots(2, n_mods, figsize=(5 * n_mods, 8))
        if n_mods == 1:
            axes = axes.reshape(2, 1)
        fig.suptitle(
            f"{metric_name} — Failed vs Passing\n"
            f"{split_name}  |  {label_name}  |  threshold={DICE_FAIL_THRESH}",
            fontsize=12, fontweight="bold"
        )
        for col, mod in enumerate(MODALITIES):
            key = f"{metric_name}_{mod}"
            fail_vals = [v[key] for v in fail_metrics.values()
                         if not np.isnan(v.get(key, np.nan))]
            pass_vals = [v[key] for v in pass_metrics.values()
                         if not np.isnan(v.get(key, np.nan))]

            _boxplot_fail_pass(axes[0][col], fail_vals, pass_vals,
                               title=mod.upper(), ylabel=metric_name)

            for group, vals in [("Failed", fail_vals), ("Passing", pass_vals)]:
                if vals:
                    arr = np.array(vals)
                    print(f"  {metric_name} {mod.upper()} {group:8s}: "
                          f"mean={arr.mean():.3f}  std={arr.std():.3f}")
            if len(fail_vals) >= 2 and len(pass_vals) >= 2:
                w1  = wasserstein_distance(np.array(fail_vals), np.array(pass_vals))
                _, p_mw = stats.mannwhitneyu(fail_vals, pass_vals, alternative="two-sided")
                print(f"  W1={w1:.3f}  "
                      f"Mann-Whitney p={p_mw:.4f}")

        plt.tight_layout()
        safe = lambda s: s.replace("/", "_").replace(" ", "_")
        out  = os.path.join(OUTPUT_DIR,
               f"quality_{safe(split_name)}_{safe(label_name)}_{safe(metric_name)}.png")
        plt.savefig(out, dpi=150)
        plt.close()
        print(f"Saved {out}")



## 7. Main Analysis Pipeline

In [ ]:

# define full analysis for each split
def run_analysis(split_name, dice_cases, png_cfgs, label_names=None):

    if label_names is None:
        label_names = ["Mean", "NCR", "Edema", "ET"]
    if isinstance(png_cfgs, DatasetConfig):
        png_cfgs = [png_cfgs]

    all_pids = list(dice_cases.keys())
    print(f"\n{'='*60}\nSplit: {split_name} ({len(all_pids)} patients)\n{'='*60}")

    all_img = load_multi_root(png_cfgs, patient_ids=all_pids)

    for label_name in label_names:
        print(f" {label_name}")
        failed_ids, passing_ids = split_by_dice(dice_cases, label_name, DICE_FAIL_THRESH)

        fail_img = {pid: all_img[pid] for pid in failed_ids  if pid in all_img}
        pass_img = {pid: all_img[pid] for pid in passing_ids if pid in all_img}
        print(f" Patient data: {len(fail_img)} failed, {len(pass_img)} passing")

        fail_metrics = compute_metrics(fail_img)
        pass_metrics = compute_metrics(pass_img)

        plot_quality_fail_vs_pass(fail_metrics, pass_metrics, split_name, label_name)


In [ ]:
# Parse  JSON result files
training_json_cases  = aggregate_cases(VAL_JSON)
test_json_cases = aggregate_cases(TEST_JSON)
ucsf_json_cases = aggregate_cases(UCSF_JSON)


In [ ]:
# Run full Analyses Training

run_analysis("Training", training_json_cases, png_cfgs=[BRATS_TRAIN_CFG, BRATS_VAL_CFG])

In [ ]:
# BraTS test
run_analysis("Test", test_json_cases, png_cfgs=BRATS_TEST_CFG)

In [ ]:
# UCSF
run_analysis("UCSF", ucsf_json_cases, png_cfgs=UCSF_CFG)